# Demo 04 — Agente completo (leitura dirigida)

04 - AGENTE COMPLETO: planejar, orquestrar e conter.
Aula 1 - Agentic Workflows (Especializacao em IA Generativa / UFPR).

Esta e a demo dos tres topicos de LEITURA DIRIGIDA (planning, orquestracao,
guardrails/avaliacao). Ela costura os padroes num pipeline unico, do jeito que
um sistema de producao faz:

  [1] PLANEJAR      - o modelo decompoe a meta em passos ordenados (JSON).
  [2] ORQUESTRAR    - dois papeis especializados trabalham sobre um ESTADO
                      compartilhado: o Analista de Dados le o banco e produz
                      fatos; o Analista de Risco decide com base neles.
                      Formalmente: S_{t+1} = delta(S_t, a_i, o_i).
  [3] CONTER        - tres camadas de guardrail sobre a saida:
                      (a) deterministica: campos obrigatorios;
                      (b) PII: nao pode vazar dado pessoal na resposta;
                      (c) LLM-as-judge: nota por rubrica, em JSON.
  [4] MEDIR         - grafico das notas por criterio.

Custo: 4 chamadas ao modelo (~10 s). Roda sobre o CLIENTE 256 - o mesmo das
demos 02 e 03. A base de credito o rotula como risco 'bom', mas o historico
de transacoes mostra tres saques no MESMO DIA em tres cidades diferentes.
O interessante e ver se o agente percebe a contradicao entre o rotulo e o
comportamento - e se o guardrail segura a decisao quando ele nao percebe.

ATENCAO DIDATICA: rode duas vezes e a DECISAO pode mudar (APROVAR / RECUSAR /
ANALISE_HUMANA) entre uma execucao e outra. Isso nao e bug: e a natureza
nao-deterministica do modelo. E exatamente por isso que a decisao nao pode
sair sem passar pela camada [3] - o guardrail e a revisao humana existem para
conter essa variabilidade, nao para elimina-la.

> **Como rodar:** abra este notebook no Google Colab e escolha *Ambiente de execução → Executar tudo*. As células já vêm na ordem certa.

> A chave da OpenRouter usada abaixo é a **chave temporária da turma**, embutida de propósito para a aula funcionar sem setup. Ela é descartável: não é uma credencial pessoal, e é revogada depois do curso.


## Ambiente (só no Colab; local com `requirements.txt` pode pular)

In [1]:
!pip install -q openai python-dotenv matplotlib

## Configuração

In [2]:
import json
import os
import re
import sqlite3
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

CHAVE_TEMPORARIA = "REMOVIDA"   # chave da aula; o ambiente sempre vence
MODELO = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.6-luna")
CHAVE  = os.getenv("OPENROUTER_API_KEY") or CHAVE_TEMPORARIA
PASTA  = Path(globals().get("__file__", ".")).resolve().parent
DB     = next(p for p in (PASTA / "dados" / "curso_financeiro.db",
                          *(a / "dados" / "curso_financeiro.db" for a in PASTA.parents),
                          PASTA / "curso_financeiro.db") if p.exists())
FIG    = PASTA / "04-agente-notas.png"
CLIENTE = 256   # o fio narrativo do curso

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=CHAVE)



def chamar_llm(sistema, usuario, tentativas=2):
    """Unico helper de rede: chama o modelo sem deixar vazar traceback."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = client.chat.completions.create(
                model=MODELO,
                messages=[{"role": "system", "content": sistema},
                          {"role": "user", "content": usuario}],
            )
            return (resp.choices[0].message.content or "").strip()
        except Exception as erro:
            print(f"[AVISO] Falha na chamada ao modelo ({tentativa}/{tentativas}): "
                  f"{type(erro).__name__}: {erro}")
            if tentativa < tentativas:
                time.sleep(2)
    sys.exit("[ERRO] Nao foi possível falar com o OpenRouter. Verifique a rede e a chave.")


def extrair_json(texto, padrao):
    """Le o primeiro objeto JSON do texto; devolve `padrao` se não houver."""
    if not texto:
        return padrao
    achado = re.search(r"\{.*\}", texto, re.DOTALL)
    if not achado:
        return padrao
    try:
        return json.loads(achado.group(0))
    except json.JSONDecodeError:
        return padrao


print(f"AGENTE COMPLETO - modelo: {MODELO} | cliente {CLIENTE}")

AGENTE COMPLETO - modelo: openai/gpt-5.6-luna | cliente 256


## [1] PLANEJAR - a meta vira uma lista ordenada de passos.

In [3]:
META = (f"Decidir se aprovamos a proposta de crédito do cliente {CLIENTE}, "
        "usando o cadastro e o historico de transacoes disponiveis no banco.")

print(f"\n[1] PLANEJAR\n    meta: {META}")

# O CATALOGO DE ACOES vai DENTRO do prompt do planejador. E a mitigacao do
# failure mode mais comum de planejamento com LLM: o modelo planejar num
# vocabulario que o executor não fala ("analisar profundamente", "consultar o
# Serasa"). Com o catalogo no prompt, o plano fica verificavel POR PROGRAMA.
CATALOGO_DE_ACOES = {
    "consultar_cliente":    "le o cadastro de crédito do solicitante no banco",
    "consultar_transacoes": "le as transacoes do solicitante no banco",
    "resumir_fatos":        "condensa cadastro e transacoes em 3 a 5 fatos objetivos",
    "emitir_parecer":       "decide APROVAR, RECUSAR ou ANALISE_HUMANA a partir dos fatos",
}
# Ordem canonica de dependencia: cada acao so roda depois do que ela le.
ORDEM_CANONICA = list(CATALOGO_DE_ACOES)

bruto = chamar_llm(
    "Voce e um planejador. Decomponha a meta em passos ordenados usando "
    "EXCLUSIVAMENTE as acoes do catalogo abaixo, uma por linha do plano. "
    'Responda APENAS com JSON: {"passos": ["acao", "acao", ...]}.\n\n'
    "Catalogo de acoes permitidas:\n"
    + "\n".join(f"- {nome}: {desc}" for nome, desc in CATALOGO_DE_ACOES.items()),
    META,
)
plano = extrair_json(bruto, {"passos": []})
propostos = plano.get("passos") or []
if not isinstance(propostos, list):
    propostos = []
propostos = [str(p).strip() for p in propostos]

# VALIDACAO DO PLANO, sem LLM nenhum: comparamos a lista de passos com a lista
# de acoes registradas. Esta e a validação barata que vem ANTES de qualquer
# LLM-as-judge - e o efeito colateral bom de ter posto o catalogo no prompt.
aceitos = [p for p in propostos if p in CATALOGO_DE_ACOES]
rejeitados = [p for p in propostos if p not in CATALOGO_DE_ACOES]
faltando = [a for a in ORDEM_CANONICA if a not in aceitos]

# O plano executado é ordenado pela dependencia, não pela ordem que o modelo
# devolveu: um plano bonito com a ordem errada continua sendo um plano errado.
passos = sorted(set(aceitos) | set(faltando), key=ORDEM_CANONICA.index)

for i, passo in enumerate(passos, 1):
    print(f"    {i}. {passo:22s} ({CATALOGO_DE_ACOES[passo]})")
if rejeitados:
    print(f"    [validação] {len(rejeitados)} passo(s) FORA do catalogo, "
          f"descartado(s): {rejeitados}")
if faltando:
    print(f"    [validação] {len(faltando)} passo(s) obrigatorio(s) ausente(s), "
          f"acrescentado(s): {faltando}")
if not rejeitados and not faltando:
    print("    [validação] plano 100% dentro do catalogo, executado como veio.")


[1] PLANEJAR
    meta: Decidir se aprovamos a proposta de crédito do cliente 256, usando o cadastro e o historico de transacoes disponiveis no banco.


    1. consultar_cliente      (le o cadastro de crédito do solicitante no banco)
    2. consultar_transacoes   (le as transacoes do solicitante no banco)
    3. resumir_fatos          (condensa cadastro e transacoes em 3 a 5 fatos objetivos)
    4. emitir_parecer         (decide APROVAR, RECUSAR ou ANALISE_HUMANA a partir dos fatos)
    [validação] plano 100% dentro do catalogo, executado como veio.


## [2] ORQUESTRAR - o executor LE O PLANO, passo a passo.

O estado e um dicionario simples: cada acao LE o que ja existe e ESCREVE sua
contribuicao. Nao ha framework, e de proposito: o padrao e o que importa.

Repare que o laco abaixo percorre `passos`. Se o plano nao fosse lido pelo
executor, ele nao seria um plano - seria um comentario caro.

In [4]:
print("\n[2] ORQUESTRAR (o executor percorre o plano validado)")

estado = {"cliente_id": CLIENTE, "plano": passos}


def acao_consultar_cliente():
    """Papel 0 (deterministico): SQL. Nao se gasta LLM com consulta."""
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    linha = con.execute(
        "SELECT cliente_id, idade, emprego, moradia, valor_credito, duracao_meses, "
        "proposito, taxa_parcela_pct, historico_credito, creditos_no_banco, risco "
        "FROM clientes WHERE cliente_id = ?", (CLIENTE,)).fetchone()
    con.close()
    if linha is None:
        sys.exit(f"[ERRO] Cliente {CLIENTE} não existe no banco.")
    estado["cadastro"] = dict(linha)
    return "cadastro carregado do banco"


def acao_consultar_transacoes():
    """Papel 0 (deterministico): SQL."""
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    linhas = con.execute(
        "SELECT data, tipo, valor, cidade, canal FROM transacoes "
        "WHERE cliente_id = ? ORDER BY data", (CLIENTE,)).fetchall()
    con.close()
    estado["transacoes"] = [dict(t) for t in linhas]
    return f"{len(estado['transacoes'])} transacoes carregadas"


def acao_resumir_fatos():
    """Papel 1: Analista de Dados. Le o estado, escreve 'fatos'."""
    estado["fatos"] = chamar_llm(
        "Voce e Analista de Dados de crédito. Liste de 3 a 5 fatos objetivos e "
        "quantitativos sobre o solicitante, em bullets curtos. Nao opine, não "
        "recomende, não conclua nada. Apenas os fatos.",
        json.dumps({"cadastro": estado["cadastro"],
                    "transacoes": estado["transacoes"]},
                   ensure_ascii=False, default=str),
    )
    return "escreveu 'fatos' no estado\n      " + estado["fatos"].replace("\n", "\n      ")


def acao_emitir_parecer():
    """Papel 2: Analista de Risco. Le 'fatos', escreve 'parecer'."""
    estado["parecer"] = chamar_llm(
        "Voce e Analista de Risco. Com base APENAS nos fatos recebidos, emita um "
        "parecer curto. Termine obrigatoriamente com uma linha exatamente no "
        "formato 'DECISAO: APROVAR' ou 'DECISAO: RECUSAR' ou 'DECISAO: ANALISE_HUMANA'.",
        f"Fatos levantados:\n{estado['fatos']}",
    )
    return "escreveu 'parecer' no estado\n      " + estado["parecer"].replace("\n", "\n      ")


EXECUTOR = {
    "consultar_cliente":    ("busca", acao_consultar_cliente),
    "consultar_transacoes": ("busca", acao_consultar_transacoes),
    "resumir_fatos":        ("papel 1: Analista de Dados", acao_resumir_fatos),
    "emitir_parecer":       ("papel 2: Analista de Risco", acao_emitir_parecer),
}

for passo in passos:
    quem, funcao = EXECUTOR[passo]
    print(f"    [{quem}] {passo}: {funcao()}")

print(f"\n    estado final contem as chaves: {sorted(estado.keys())}")


[2] ORQUESTRAR (o executor percorre o plano validado)
    [busca] consultar_cliente: cadastro carregado do banco
    [busca] consultar_transacoes: 3 transacoes carregadas


    [papel 1: Analista de Dados] resumir_fatos: escreveu 'fatos' no estado
      - Cliente de 27 anos, com emprego classificado como não qualificado e moradia própria.
      - Crédito solicitado de 7.418, com prazo de 60 meses e taxa de parcela de 1%.
      - Histórico de crédito registra atraso no passado e 1 crédito no banco.
      - Foram realizados 3 saques em 11/06/2025, totalizando 2.350.
      - Os saques ocorreram em São Paulo (800), Rio de Janeiro (650) e Belo Horizonte (900), todos via ATM.


    [papel 2: Analista de Risco] emitir_parecer: escreveu 'parecer' no estado
      Há fatores de risco relevantes: histórico de atraso, emprego não qualificado, prazo longo de 60 meses e três saques em cidades distintas no mesmo dia, totalizando 2.350. A moradia própria e o relacionamento bancário são fatores positivos, mas insuficientes para eliminar as inconsistências. Recomenda-se verificação adicional da origem e finalidade dos saques e da capacidade de pagamento.
      
      DECISAO: ANALISE_HUMANA

    estado final contem as chaves: ['cadastro', 'cliente_id', 'fatos', 'parecer', 'plano', 'transacoes']


## [3] CONTER - tres camadas de guardrail sobre o parecer.

In [5]:
print("\n[3] GUARDRAILS SOBRE A SAIDA")

parecer = estado["parecer"]

# (a) deterministico: a decisao esta no formato combinado?
achado = re.search(r"DECISAO:\s*(APROVAR|RECUSAR|ANALISE_HUMANA)", parecer)
decisao = achado.group(1) if achado else None
print(f"    (a) formato da decisao ....... {decisao if decisao else 'AUSENTE -> bloquear'}")

# (b) PII: o parecer não pode conter CPF, e-mail ou telefone.
PADROES_PII = {
    "CPF": r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
    "e-mail": r"\b[\w.+-]+@[\w-]+\.[\w.]+\b",
    "telefone": r"\b\(?\d{2}\)?\s?9?\d{4}-?\d{4}\b",
}
vazamentos = [nome for nome, padrao in PADROES_PII.items() if re.search(padrao, parecer)]
print(f"    (b) vazamento de PII ......... "
      f"{'NENHUM' if not vazamentos else 'ENCONTRADO: ' + ', '.join(vazamentos)}")

# (c) LLM-as-judge: nota de 0 a 5 por criterio, em JSON (parse robusto).
RUBRICA = ["fundamentacao nos fatos", "clareza para o gerente", "aderencia ao formato pedido"]
bruto = chamar_llm(
    "Voce e um avaliador rigoroso. Atribua nota INTEIRA de 0 a 5 para cada "
    "criterio. Responda APENAS com JSON no formato "
    '{"notas": {"criterio": nota, ...}, "comentario": "uma frase"}.',
    f"Criterios: {RUBRICA}\n\nFatos:\n{estado['fatos']}\n\nParecer avaliado:\n{parecer}",
)
avaliacao = extrair_json(bruto, {"notas": {}, "comentario": "[avaliacao indisponivel]"})
notas = {k: v for k, v in (avaliacao.get("notas") or {}).items()
         if isinstance(v, (int, float))}

print("    (c) LLM-as-judge:")
for criterio, nota in notas.items():
    # A nota vem de um LLM: pode chegar fora da faixa 0-5. Limitamos para que
    # a barra continue legivel mesmo se o modelo extrapolar a escala pedida.
    nota = max(0, min(5, int(nota)))
    barra = "#" * nota + "." * (5 - nota)
    print(f"          {criterio:32s} [{barra}] {nota}/5")
print(f"          comentario: {avaliacao.get('comentario', '')}")

# A porta final: so passa se as tres camadas aprovarem.
media = sum(notas.values()) / len(notas) if notas else 0
aprovado = bool(decisao) and not vazamentos and media >= 3
print(f"\n    >> PORTA FINAL: {'LIBERADO' if aprovado else 'RETIDO PARA REVISAO HUMANA'} "
      f"(media {media:.1f}/5)")


[3] GUARDRAILS SOBRE A SAIDA
    (a) formato da decisao ....... ANALISE_HUMANA
    (b) vazamento de PII ......... NENHUM


    (c) LLM-as-judge:
          fundamentacao nos fatos          [#####] 5/5
          clareza para o gerente           [#####] 5/5
          aderencia ao formato pedido      [#####] 5/5
          comentario: O parecer utiliza corretamente os fatos apresentados, comunica os riscos e recomendações com clareza e inclui a decisão no formato esperado.

    >> PORTA FINAL: LIBERADO (media 5.0/5)


## [4] MEDIR - o grafico das notas.

In [6]:
if notas:
    criterios = list(notas.keys())
    valores = [notas[c] for c in criterios]
    fig, ax = plt.subplots(figsize=(7, 3.2))
    cores = ["#2c5282" if v >= 3 else "#c0392b" for v in valores]
    ax.barh(criterios, valores, color=cores)
    ax.set_xlim(0, 5)
    ax.set_xlabel("nota (0 a 5)")
    ax.set_title(f"Avaliacao do parecer - cliente {CLIENTE} (media {media:.1f})")
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(FIG, dpi=110)
    plt.close(fig)
    print(f"\n[grafico salvo] {FIG.name}")

print("LICAO: o agente util não e o mais esperto - e o que tem porta de saida.")
print("Planejar da rumo; orquestrar divide o trabalho; guardrail decide o que sai.")


[grafico salvo] 04-agente-notas.png
LICAO: o agente util não e o mais esperto - e o que tem porta de saida.
Planejar da rumo; orquestrar divide o trabalho; guardrail decide o que sai.
